# MI300X decode: roofline and vLLM comparison

Run inside the `rocm` container on the droplet (VS Code: Remote-SSH to `mi300x`, then *Dev Containers: Attach to Running Container* → `rocm`, open `/shared-docker/llama70b-mi300x`).

In [ ]:
import os, subprocess, sys
os.chdir('/shared-docker/llama70b-mi300x' if os.path.isdir('/shared-docker/llama70b-mi300x') else '..')
import pandas as pd
import matplotlib.pyplot as plt
MEASURED_BW = 3.6  # TB/s, from scripts/bench_bandwidth.py
SPEC_BW = 5.3

## Per-kernel bandwidth on 70B decode shapes

Re-run the benchmark (about a minute) or plot the saved CSV.

In [ ]:
RERUN = False
csv = 'results/kernels.csv'
if RERUN or not os.path.exists(csv):
    subprocess.run([sys.executable, 'scripts/bench_kernels.py', '--batches', '1', '8', '32', '--ctx', '1024', '8192', '--csv', csv], check=True)
df = pd.read_csv(csv)
df['label'] = df['op'] + '\nM=' + df['M'].astype(str) + ('' if 'N' not in df else '')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
gemm = df[df.op.str.contains('gemm|scaled_mm')].copy()
gemm['shape'] = gemm.op.str.split('.').str[1]
gemm['impl'] = gemm.op.str.split('.').str[0].map({'fp8_gemm': 'Triton (this repo)', 'scaled_mm': 'hipBLASLt'})
piv = gemm.pivot_table(index=['shape', 'M'], columns='impl', values='tbps')
piv.plot.bar(ax=ax, width=0.8)
ax.axhline(MEASURED_BW, ls='--', c='k', lw=1, label=f'measured peak {MEASURED_BW} TB/s')
ax.axhline(SPEC_BW, ls=':', c='gray', lw=1, label=f'spec {SPEC_BW} TB/s')
ax.set_ylabel('achieved TB/s'); ax.set_xlabel('')
ax.set_title('FP8 GEMM, Llama-3.3-70B decode shapes')
ax.legend(); plt.xticks(rotation=0); plt.tight_layout()

In [ ]:
att = df[df.op == 'decode_attn']
fig, ax = plt.subplots(figsize=(7, 4))
for ctx, g in att.groupby('N'):
    ax.plot(g.M, g.tbps, marker='o', label=f'ctx={ctx}')
ax.axhline(MEASURED_BW, ls='--', c='k', lw=1, label='measured peak')
ax.set_xscale('log', base=2); ax.set_xlabel('batch'); ax.set_ylabel('achieved TB/s')
ax.set_title('Paged decode attention'); ax.legend(); plt.tight_layout()

## End-to-end decode vs vLLM

Numbers from `results/compare_vllm.log` (64-token window, best of 3).

In [ ]:
e2e = pd.DataFrame({
    'batch': [1, 8, 32],
    'this repo': [27.77, 28.92, 31.98],
    'vLLM default': [28.33, 29.06, 31.05],
    'vLLM + AITER': [27.14, 27.35, 29.70],
}).set_index('batch')
floor = 72.7e9 / (MEASURED_BW * 1e12) * 1e3  # ms to stream the weights once
ax = e2e.plot.bar(figsize=(7, 4), width=0.8)
ax.axhline(floor, ls='--', c='k', lw=1, label=f'weight-streaming floor {floor:.1f} ms')
ax.set_ylabel('ms / decode step (lower is better)'); ax.legend(loc='lower right')
ax.set_title('Llama-3.3-70B FP8, 1x MI300X'); plt.xticks(rotation=0); plt.tight_layout()
(1000 / e2e).mul(e2e.index, axis=0).round(0).rename(columns=lambda c: c + ' tok/s')